# 07 — Simulated production: replay of Jul – Sep 2026

**Question:** if the model had run every night from 1 July to 30 September 2026, without being retrained, would it have stayed good, and would its intervals have stayed calibrated?

The replay (`swissdelay.pipeline.replay`) scores each day with the **frozen champion**: XGBoost `full_network_plus`, trained on Aug 2025 – Apr 2026 and exported in `models/champion/`, verified to reproduce the training-time predictions exactly. Each day it:

1. scores the day's prediction points: P50 plus three P10–P90 intervals;
   - `static`: conformal offsets calibrated on April (as on the test month);
   - `rolling`: offsets recomputed per current-delay bucket on the **last 14 replayed days**, symmetric;
   - `rolling_tails`: same window, one offset per tail;
2. compares with the realised Δd;
3. checks drift (PSI against training) and raises alerts;
4. only then adds the day to the rolling window, so a day never calibrates itself.

**This is a replay, not live inference.** Features were built with the same code and leakage rules, month by month, and read one day at a time.

1. Accuracy over the quarter
2. Prediction intervals: the three strategies
3. Drift
4. Alerts
5. Findings and decisions

In [ ]:
# 0. Setup
import numpy as np
import pandas as pd
import plotly.express as px

from swissdelay import config
from swissdelay.models import quantile as qt
from swissdelay.pipeline.replay import REPLAY_DIR, STRATEGIES, summary
from swissdelay.viz import GRAY, SERIES, use_style

use_style()
pd.set_option("display.float_format", "{:,.3f}".format)
m = pd.read_parquet(REPLAY_DIR / "daily_metrics.parquet")
m["operating_day"] = pd.to_datetime(m["operating_day"])
m["month"] = m["operating_day"].dt.strftime("%Y-%m")
preds = pd.concat([pd.read_parquet(p) for p in sorted(REPLAY_DIR.glob("predictions_*.parquet"))], ignore_index=True)
dis_days = m.loc[m["is_disruption_day"], "operating_day"].drop_duplicates()
print(f"{m['operating_day'].nunique()} days, {len(preds):,} scored points, {len(dis_days)} disruption days")


def shade_disruptions(fig):
    for d in dis_days:
        fig.add_vrect(x0=d - pd.Timedelta(hours=12), x1=d + pd.Timedelta(hours=12), fillcolor=GRAY,
                      opacity=0.15, line_width=0)
    return fig

## 1. Accuracy over the quarter

In [ ]:
# 1.1 Summary by horizon, normal vs disruption days (point-weighted)
s = summary(m)
s["skill_%"] = 100 * (s["mae"] / s["mae_historical"] - 1)
s[["horizon_min", "disruption_day", "days", "points", "mae", "mae_historical", "skill_%"]]

In [ ]:
# 1.2 By month: is there any slow degradation?
def wavg(g, col):
    return np.average(g[col], weights=g["n"])

monthly = (m.groupby(["horizon_min", "month"])
             .apply(lambda g: pd.Series({"mae": wavg(g, "mae"), "mae_historical": wavg(g, "mae_historical"),
                                         "disruption_days": int(g["is_disruption_day"].sum())}),
                    include_groups=False)
             .reset_index())
monthly["skill_%"] = 100 * (monthly["mae"] / monthly["mae_historical"] - 1)
reference = pd.DataFrame({"horizon_min": [15, 30, 60], "May (validation)": [0.701, 0.843, 1.005],
                          "June (test)": [0.805, 0.990, 1.200]})
display(monthly.pivot(index="month", columns="horizon_min", values=["mae", "skill_%"]).round(3))
reference

In [ ]:
# 1.3 Chart: daily MAE of the champion and the historical baseline (disruption days shaded)
d = m.melt(id_vars=["operating_day", "horizon_min"], value_vars=["mae", "mae_historical"],
           var_name="model", value_name="MAE")
d["model"] = d["model"].map({"mae": "champion", "mae_historical": "historical median"})
fig = px.line(d, x="operating_day", y="MAE", color="model", facet_row="horizon_min",
              color_discrete_map={"champion": SERIES[0], "historical median": SERIES[1]},
              title="Daily MAE, Jul – Sep 2026 (frozen model; shaded: disruption days)", height=700)
fig.update_yaxes(matches=None, title=None)
fig.update_layout(xaxis_title=None, legend_title=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("horizon_min=", "") + " min"))
shade_disruptions(fig).show()

In [ ]:
# 1.4 Skill vs historical day by day: does the gain shrink over time? (linear trend per horizon)
for h, g in m.groupby("horizon_min"):
    x = (g["operating_day"] - g["operating_day"].min()).dt.days.to_numpy()
    slope = np.polyfit(x, 100 * (1 - g["skill_vs_historical"].to_numpy()) - 100, 1)[0]
    print(f"h={h}: skill vs historical, mean {100 * g['skill_vs_historical'].mean():.1f} %, "
          f"trend {30 * slope:+.2f} points per 30 days")

> **Finding 1.1–1.4 (accuracy, frozen model, 92 days, 2.79 M scored points).**
> - **The gain over the historical median holds for the whole quarter:**
>   - normal days: −22.0 / −21.1 / −21.1 % at 15 / 30 / 60 min;
>   - disruption days: −20.4 / −19.9 / −20.2 %.
>
>   That's the same order as on May (−21 %) and June (−19 %).
> - **No ageing.**
>   - Monthly MAE at 15 min is 0.712 (Jul), 0.720 (Aug) and 0.674 (Sep), and the gain over the historical baseline stays at −20.4 to −22.0 % every month and horizon.
>   - The fitted trend of the daily gain is −0.08 / −0.37 / +0.50 points per 30 days, i.e. flat. A model trained up to April 2026 is still as good five months later.
> - **The level of the error follows the network, not the model's age.** The summer months (MAE 0.67–0.72 at 15 min) sit between the calm May (0.70) and the disrupted June (0.81). Disruption days cost about +0.09 / +0.14 / +0.20 min of MAE.

## 2. Prediction intervals

In [ ]:
# 2.1 Coverage, tails and width of the three strategies (point-weighted)
rows = []
for (h, dis), g in m.groupby(["horizon_min", "is_disruption_day"]):
    for st in STRATEGIES:
        rows.append({"horizon_min": h, "disruption_day": dis, "strategy": st,
                     "coverage": wavg(g, f"coverage_{st}"), "below_p10": wavg(g, f"below_{st}"),
                     "above_p90": wavg(g, f"above_{st}"), "median_width": wavg(g, f"width_{st}")})
iv = pd.DataFrame(rows)
iv.pivot_table(index=["horizon_min", "disruption_day"], columns="strategy",
               values=["coverage", "below_p10", "above_p90", "median_width"]).round(3)

In [ ]:
# 2.2 By current delay (all days), from the scored points
preds["d0_bucket"] = pd.Categorical(preds["d0_bucket"], categories=qt.D0_BUCKETS[1], ordered=True)
rows = []
for (h, b), g in preds.groupby(["horizon_min", "d0_bucket"], observed=True):
    y = g["y"].to_numpy()
    for st in STRATEGIES:
        lo, hi = g[f"lo_{st}"].to_numpy(), g[f"hi_{st}"].to_numpy()
        rows.append({"horizon_min": h, "d0_bucket": b, "strategy": st, "n": len(g),
                     "coverage": np.mean((y >= lo) & (y <= hi)), "below_p10": np.mean(y < lo),
                     "above_p90": np.mean(y > hi)})
pd.DataFrame(rows).pivot_table(index=["horizon_min", "d0_bucket"], columns="strategy",
                               values=["coverage", "above_p90"], observed=True).round(3)

In [ ]:
# 2.3 Chart: daily coverage of the three strategies at 60 min (target 0.80)
d = m[m["horizon_min"] == 60].melt(id_vars="operating_day", value_vars=[f"coverage_{s}" for s in STRATEGIES],
                                    var_name="strategy", value_name="coverage")
d["strategy"] = d["strategy"].str.replace("coverage_", "")
fig = px.line(d, x="operating_day", y="coverage", color="strategy",
              color_discrete_map={"static": SERIES[1], "rolling": SERIES[3], "rolling_tails": SERIES[0]},
              title="Daily P10–P90 coverage at 60 min (target 80 %; shaded: disruption days)")
fig.add_hline(y=0.8, line_dash="dot", line_color=GRAY)
fig.update_layout(yaxis_tickformat=".0%", xaxis_title=None, legend_title=None)
shade_disruptions(fig).show()

> **Finding 2.1–2.3 (prediction intervals).**
> - **`static` (April offsets) under-covers all quarter:** 76.5–78.4 % (as on June), with the misses in the upper tail (11.3–14.0 % above P90).
> - **`rolling` (last 14 days, symmetric) reaches the target on normal days** (80.0 / 80.0 / 80.1 %) and 78.6–79.1 % on disruption days. But the tails stay unbalanced: 10.8–13.4 % above P90 and 8.0–9.1 % below P10.
> - **`rolling_tails` (last 14 days, one offset per tail) is the best:**
>   - coverage is the same as `rolling` (80.0 % on normal days, 78.7–79.1 % on disruption days);
>   - **both tails are balanced**: 10.0 % below P10 and 9.9 % above P90 on normal days, 10.9–11.5 % above on disruption days;
>   - it is calibrated in **every current-delay bucket** (79.8–80.3 %; 10.0–10.2 % above P90 everywhere), including major delays, where `rolling` leaves 12.6–12.9 % above.
> - **Cost:** intervals are 5–10 % wider than `static` (median width 1.88 / 2.33 / 2.79 min on normal days, vs 1.79 / 2.17 / 2.54).
> - The remaining shortfall on disruption days (about 1 point) is expected. The window is mostly made of normal days, and a disrupted day can only be learned from after it has happened.

## 3. Drift

In [ ]:
# 3.1 PSI against the training distribution (< 0.1 stable, 0.1–0.25 moderate, > 0.25 large)
psi_cols = [c for c in m.columns if c.startswith("psi_")]
display(m.groupby("horizon_min")[psi_cols].agg(["mean", "max"]).round(3))
d = m[m["horizon_min"] == 15].melt(id_vars="operating_day", value_vars=psi_cols, var_name="feature", value_name="PSI")
fig = px.line(d, x="operating_day", y="PSI", color="feature", title="Daily drift (PSI vs training), 15 min",
              color_discrete_sequence=SERIES)
fig.add_hline(y=0.1, line_dash="dot", line_color=GRAY)
fig.add_hline(y=0.25, line_dash="dash", line_color=GRAY)
fig.update_layout(xaxis_title=None, legend_title=None)
fig.show()

> **Finding 3.1 (drift).**
> - All input distributions stay close to training. The mean PSI is 0.011–0.015 for current delay, 0.001–0.002 for hour and 0.006–0.011 for station traffic.
> - The worst single day reaches 0.079, below the 0.1 "stable" threshold. **No drift alert all quarter**, consistent with the flat accuracy in 1.4.

## 4. Alerts

In [ ]:
# 4.1 Every alert of the quarter
a = m[m["alerts"] != ""][["operating_day", "horizon_min", "alerts", "mae", "mae_historical", "is_disruption_day",
                          "coverage_rolling_tails"]]
print(f"{len(a)} alerts on {a['operating_day'].nunique()} days")
a

> **Finding 4.1 (alerts).**
> - **Only 5 alerts on 3 days, all `mae_jump`, all in July:** 9 July at 30 and 60 min; 14 July (a disruption day) at 30 and 60 min; 16 July (a disruption day) at 60 min.
> - On each of them, MAE was more than 25 % above its median of the previous 14 days but **still 18–19 % below the historical baseline**: bad days for the network, not failures of the model.
> - There were no low-coverage alerts (the lowest daily coverage of `rolling_tails` on an alert day was 75.5 %, above the 70 % threshold) and no drift alerts.
> - With 5 alerts in 276 day-horizons, the thresholds are not noisy.

## 5. Findings and decisions

### Simulated production, Jul – Sep 2026 (92 days, frozen model)

| | 15 min | 30 min | 60 min |
|---|---|---|---|
| MAE, normal days | 0.690 | 0.865 | 1.056 |
| MAE, disruption days | 0.780 | 1.002 | 1.253 |
| vs historical median (normal / disruption) | −22.0 / −20.4 % | −21.1 / −19.9 % | −21.1 / −20.2 % |
| P10–P90 coverage, `rolling_tails` (normal / disruption) | 80.0 / 79.1 % | 80.0 / 79.0 % | 80.1 / 78.7 % |
| P10–P90 coverage, `static` (normal / disruption) | 78.4 / 77.5 % | 77.9 / 76.8 % | 78.1 / 76.5 % |

### Answers

- **Does the model degrade without retraining?** Not over five months after the end of training: the gain over the historical baseline is flat (trend within ±0.5 points per 30 days), and drift stays below the "stable" threshold every day.
- **Can the intervals be kept calibrated?** Yes, by **recalibrating every night on the last 14 days, with one offset per tail**. This gives 80 % coverage with 10 % in each tail on normal days, in every delay bucket, at the cost of 5–10 % wider intervals.
- **Does the monitoring work?** Yes: 5 alerts in 92 days, all on genuinely bad days, and none false.

### Decisions

| # | Topic | Decision | Evidence |
|---|---|---|---|
| 1 | Point forecast | XGBoost `full_network_plus`, frozen; no retraining needed over the replayed quarter | 1.1–1.4, 3.1 |
| 2 | Intervals in production | **`rolling_tails`**: each night, per horizon and current-delay bucket, one conformal offset per tail from the last 14 days (fallback to the April offsets during the first 7 days) | 2.1–2.3 |
| 3 | Retraining | No scheduled retraining. Retrain if the gain over the historical baseline falls by more than 3 points over 30 days, or if drift (PSI) exceeds 0.25 for a week. The 2 % promotion rule then decides whether the new model replaces the old one | 1.4, 3.1 |
| 4 | Alerts | Thresholds kept (PSI > 0.25, coverage < 70 %, MAE > 1.25 × 14-day median) | 4.1 |
| 5 | Next | API (`/predict` with the `rolling_tails` interval, `/metrics/daily`) and dashboard, in Docker; then the write-up and website | — |